# 06 — Taxonomy group analysis (supplementary, no rerun)
Confirmation experiments built on nb01's saved outputs only: individual-confidence collapse, full-simplex JS saturation, subtree-projection de-saturation, and the decisive comparison of GUARD group scores against trivial per-item aggregation. Reads `item_meta.parquet`, `pi_memmap.npy`, `label_classes.npy`, `category_mapping.csv`; retrains nothing.

In [1]:
# Notebook: 06_taxonomy_group_analysis
# Supplementary confirmation analyses. Reads ONLY the artifacts saved by nb01
# (item_meta.parquet, pi_memmap.npy, label_classes.npy) + category_mapping.csv.
# No retraining, no rerun of nb00-05 required. Grayscale figures, dpi 600, PNG+PDF, no captions.
import os, csv, sys
import numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
sys.path.append(os.path.join("..", "src"))
from guard_core import entropy, js_divergence, guard_by_group
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, average_precision_score

sns.set_theme(style="whitegrid", context="paper", font_scale=1.1)
plt.rcParams["axes.edgecolor"]="0.2"; plt.rcParams["axes.linewidth"]=0.8
plt.rcParams["font.family"]="DejaVu Sans"
GREYS=["#111111","#555555","#888888","#bbbbbb","#dddddd"]
FIG=os.path.join("..","results","figures"); TAB=os.path.join("..","results","tables")
os.makedirs(FIG,exist_ok=True); os.makedirs(TAB,exist_ok=True)
def savefig(fig,name):
    for ext in ("png","pdf"): fig.savefig(os.path.join(FIG,f"{name}.{ext}"),dpi=600,bbox_inches="tight")
    plt.close(fig)


In [2]:
# --- load saved artifacts + taxonomy (join confirmed: raw category id == category_label) ---
DATA_DIR=os.path.join("..","data")
meta=pd.read_parquet(os.path.join(DATA_DIR,"item_meta.parquet"))
pi=np.load(os.path.join(DATA_DIR,"pi_memmap.npy"),mmap_mode="r"); N,K=pi.shape
classes=np.load(os.path.join(DATA_DIR,"label_classes.npy"))     # dense id -> raw category_label

p=os.path.join(DATA_DIR,"category_mapping.csv")
with open(p,encoding="utf-8",errors="replace") as f: hd=[f.readline() for _ in range(6)]
try: sep=csv.Sniffer().sniff("".join(hd),delimiters=[",","\t",";","|"]).delimiter
except Exception: sep="\t"
cmap=pd.read_csv(p,sep=sep,engine="python"); lab2name=dict(zip(cmap.category_label,cmap.category_name))

# per-dense-leaf taxonomy path parts
leaf_parts=[([s.strip() for s in lab2name.get(int(classes[d]),"").split(">")]
             if lab2name.get(int(classes[d]),"") else []) for d in range(K)]

noisy_dense=meta["noisy_id"].values
mis=meta["is_misregistered"].values
valid=meta["clean_id"].values>=0
p_noisy=meta["p_noisy"].values
ent=meta["entropy"].values
print(f"N={N:,}  K={K:,}  misregistration rate={mis[valid].mean():.3%}")


N=502,310  K=5,691  misregistration rate=14.748%


In [3]:
# --- Finding 1: individual confidence collapses (recompute AUROC/AUPRC) ---
rows=[]
for name,s in [("1 - p_noisy",1.0-p_noisy),("entropy",ent)]:
    a=roc_auc_score(mis[valid],s[valid]); ap=average_precision_score(mis[valid],s[valid])
    rows.append((name,a,ap))
# margin needs top-prob; compute in chunks to stay light on the memmap
top=np.empty(N,dtype=np.float32)
for i in range(0,N,20000):
    top[i:i+20000]=np.asarray(pi[i:i+20000],dtype=np.float32).max(1)
mg=top-p_noisy
a=roc_auc_score(mis[valid],mg[valid]); ap=average_precision_score(mis[valid],mg[valid])
rows.append(("margin",a,ap))
indiv=pd.DataFrame(rows,columns=["signal","AUROC","AUPRC"]).sort_values("AUROC",ascending=False)
indiv.to_csv(os.path.join(TAB,"t06_individual_detection.csv"),index=False)
print(indiv.to_string(index=False))

fig,ax=plt.subplots(figsize=(4.8,3.2))
x=np.arange(len(indiv))
ax.bar(x,indiv["AUROC"],color=GREYS[1],edgecolor="black",linewidth=0.6)
ax.axhline(0.5,color="black",ls=":",lw=1)
ax.set_xticks(x); ax.set_xticklabels(indiv["signal"],rotation=20,ha="right")
ax.set_ylabel("AUROC (misregistration)"); ax.set_ylim(0.45,0.75)
savefig(fig,"f06a_individual_detection")
print("saved f06a_individual_detection.{png,pdf}")


     signal    AUROC    AUPRC
    entropy 0.534676 0.157245
1 - p_noisy 0.525740 0.156884
     margin 0.511861 0.151228
saved f06a_individual_detection.{png,pdf}


In [4]:
# --- Finding 2a: full-simplex JS grouping SATURATES (leaf-parent grouping) ---
a=noisy_dense
# item-level parent path via each item's noisy (registered) leaf
item_parts=[leaf_parts[d] for d in noisy_dense]
def item_parent_at(depth):
    return np.array([" > ".join(pp[:depth]) if len(pp)>=depth else None for pp in item_parts],dtype=object)

sweep=[]
for DEPTH in [2,3,4]:
    gid=item_parent_at(DEPTH); s=pd.Series(gid); sizes=s.value_counts()
    keep=set(sizes[(sizes>=30)&(sizes<=50000)].index)-{None}; mask=s.isin(keep).values
    g=pd.DataFrame(guard_by_group(pi,a[mask],gid[mask],K,min_size=30))
    nb=pd.Series(mis[mask&valid]).groupby(gid[mask&valid]).mean()
    g["true_noise_rate"]=g["group"].map(nb); g=g.dropna(subset=["true_noise_rate"])
    sweep.append(dict(depth=DEPTH,n_groups=len(g),C_full_mean=round(g.C.mean(),3),
                      C_full_std=round(g.C.std(),3),
                      spearman_C=round(spearmanr(g.C,g.true_noise_rate)[0],3)))
sweep=pd.DataFrame(sweep); sweep.to_csv(os.path.join(TAB,"t06_fulljs_sweep.csv"),index=False)
print("full-simplex JS (C_full) — saturated, no signal:"); print(sweep.to_string(index=False))


full-simplex JS (C_full) — saturated, no signal:
 depth  n_groups  C_full_mean  C_full_std  spearman_C
     2         7        0.936       0.040       0.214
     3        81        0.967       0.077      -0.110
     4       663        0.928       0.157      -0.045


In [5]:
# --- Finding 2b/3: subtree PROJECTION de-saturates C; compare vs trivial baselines ---
def analyze(DEPTH, want_df=False):
    leaf_parent=np.array([" > ".join(pp[:DEPTH]) if len(pp)>=DEPTH else None for pp in leaf_parts],dtype=object)
    item_parent=leaf_parent[noisy_dense]
    S_of={}
    for d in range(K):
        pk=leaf_parent[d]
        if pk is not None: S_of.setdefault(pk,[]).append(d)
    sizes=pd.Series(item_parent).value_counts()
    keep=[k for k,v in sizes.items() if k is not None and 30<=v<=50000]
    recs=[]
    for pk in keep:
        idx=np.where(item_parent==pk)[0]; S=np.array(S_of[pk]); vi=valid[idx]
        if len(S)<2 or not vi.any(): continue
        pbar=np.asarray(pi[idx],dtype=np.float64).mean(0)
        q=np.bincount(noisy_dense[idx],minlength=K).astype(float); q/=q.sum()
        pS=pbar[S]; out_mass=float(1-pS.sum())
        if pS.sum()<=0: continue
        pSn=pS/pS.sum(); qSn=q[S]/q[S].sum()
        r=np.clip(pSn-qSn,0,None); kap=0.0 if r.sum()<=1e-12 else 1-entropy(r/r.sum())/np.log(len(S))
        s_ind=1.0-p_noisy[idx]
        recs.append(dict(group=pk,n=len(idx),true_noise=mis[idx][vi].mean(),
            mean_1mp=float(s_ind.mean()), p90_1mp=float(np.quantile(s_ind,0.9)), mean_ent=float(ent[idx].mean()),
            C_full=js_divergence(q,pbar), C_proj=js_divergence(qSn,pSn),
            out_mass=out_mass, Cproj_kappa=js_divergence(qSn,pSn)*kap))
    g=pd.DataFrame(recs).dropna(subset=["true_noise"])
    return g

# decisive comparison at DEPTH=4
g4=analyze(4); g4.to_csv(os.path.join(TAB,"t06_group_score_comparison_d4.csv"),index=False)
lo,hi=g4.true_noise.quantile(1/3),g4.true_noise.quantile(2/3)
sub=g4[(g4.true_noise<=lo)|(g4.true_noise>=hi)].copy(); y=(sub.true_noise>=hi).astype(int).values
cols=["mean_1mp","p90_1mp","mean_ent","C_full","C_proj","out_mass","Cproj_kappa"]
comp=[]
for c in cols:
    rho=spearmanr(g4[c],g4.true_noise)[0]; au=roc_auc_score(y,sub[c].values); au=max(au,1-au)
    comp.append(dict(score=c,kind=("baseline" if c in ("mean_1mp","p90_1mp","mean_ent") else "GUARD"),
                     spearman=round(rho,3),auroc_hi_lo=round(au,3)))
comp=pd.DataFrame(comp); comp.to_csv(os.path.join(TAB,"t06_comparison_summary_d4.csv"),index=False)
print(f"DEPTH=4  groups={len(g4)}  (hi/lo tercile n={len(sub)})"); print(comp.to_string(index=False))


DEPTH=4  groups=475  (hi/lo tercile n=318)
      score     kind  spearman  auroc_hi_lo
   mean_1mp baseline     0.130        0.583
    p90_1mp baseline     0.210        0.622
   mean_ent baseline     0.177        0.613
     C_full    GUARD     0.131        0.583
     C_proj    GUARD     0.194        0.620
   out_mass    GUARD     0.123        0.579
Cproj_kappa    GUARD     0.195        0.623


In [6]:
# --- Figures: (B) C_full vs C_proj de-saturation ; (C) GUARD vs baseline comparison ---
# (B) distribution shift: full-JS saturates near 1, projected spreads low
figB,ax=plt.subplots(figsize=(5.2,3.4))
sns.kdeplot(g4["C_full"],ax=ax,color=GREYS[0],fill=True,alpha=0.3,label="C_full (global JS)")
sns.kdeplot(g4["C_proj"],ax=ax,color=GREYS[2],fill=True,alpha=0.3,label="C_proj (subtree)")
ax.set_xlabel("claim-belief divergence"); ax.set_ylabel("density"); ax.legend(frameon=False)
savefig(figB,"f06b_C_desaturation")

# (C) group-score comparison (AUROC hi vs lo); baselines light, GUARD dark
figC,ax=plt.subplots(figsize=(5.8,3.4))
order=comp.copy()
colors=[GREYS[3] if k=="baseline" else GREYS[0] for k in order["kind"]]
ax.bar(range(len(order)),order["auroc_hi_lo"],color=colors,edgecolor="black",linewidth=0.6)
ax.axhline(0.5,color="black",ls=":",lw=1)
ax.set_xticks(range(len(order))); ax.set_xticklabels(order["score"],rotation=30,ha="right")
ax.set_ylabel("AUROC (high vs low noise group)"); ax.set_ylim(0.5,0.7)
savefig(figC,"f06c_group_comparison")
print("saved f06b_C_desaturation.{png,pdf}, f06c_group_comparison.{png,pdf}")
print("\nAll supplementary tables/figures written to ../results/. No retraining was performed.")


saved f06b_C_desaturation.{png,pdf}, f06c_group_comparison.{png,pdf}

All supplementary tables/figures written to ../results/. No retraining was performed.
